In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor


# Project root
REPO = Path.cwd().parent

# Dataset
DATA_PATH = REPO / "data" / "processed" / "ml_training_dataset.csv"

df = pd.read_csv(DATA_PATH)

df["datetime"] = pd.to_datetime(df["datetime"])

# Time features
df["hour"] = df["datetime"].dt.hour
df["month"] = df["datetime"].dt.month
df["day_of_year"] = df["datetime"].dt.dayofyear

print("Dataset shape:", df.shape)
print("Stations:", df["station_id"].unique())

Dataset shape: (36538, 29)
Stations: <StringArray>
['BHATSANAGAR_1', 'JALNA_2', 'PAUD_1', 'SONGE_BANGE', 'SUKSALE',
 'YELDARI_DAM_1']
Length: 6, dtype: str


In [2]:
feature_cols = [
    "latitude",
    "longitude",
    "era5_temperature",
    "era5_dewpoint_temperature",
    "era5_u10",
    "era5_v10",
    "era5_relative_humidity",
    "era5_wind_speed",
    "era5_wind_direction",
    "station_elevation_m",
    "era5_elevation_m",
    "elevation_difference_m",
    "physics_temperature",
    "physics_relative_humidity",
    "physics_wind_speed",
    "humidity_clamp_flag",
    "hour",
    "month",
    "day_of_year",
]

target_cols = {
    "temperature": "temperature_residual",
    "humidity": "humidity_residual",
    "wind": "wind_speed_residual",
}

X = df[feature_cols]
groups = df["station_id"]

print("Features:", len(feature_cols))
print("Samples:", len(X))
print("Groups:", groups.nunique())

Features: 19
Samples: 36538
Groups: 6


In [3]:
param_distributions = {
    "n_estimators": [200, 300, 500, 700, 1000],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 2, 5, 10],
}

print("Hyperparameters to tune:")
for param, values in param_distributions.items():
    print(f"{param}: {values}")

Hyperparameters to tune:
n_estimators: [200, 300, 500, 700, 1000]
max_depth: [3, 4, 5, 6, 8, 10]
learning_rate: [0.01, 0.03, 0.05, 0.08, 0.1]
min_child_weight: [1, 3, 5, 10]
subsample: [0.7, 0.8, 0.9, 1.0]
colsample_bytree: [0.7, 0.8, 0.9, 1.0]
reg_alpha: [0, 0.01, 0.1, 1]
reg_lambda: [1, 2, 5, 10]


In [4]:
# GroupKFold keeps all observations from the same station together
group_kfold = GroupKFold(n_splits=5)

print("Number of folds:", group_kfold.get_n_splits())

for fold, (train_idx, val_idx) in enumerate(
    group_kfold.split(X, groups=groups), start=1
):
    train_stations = groups.iloc[train_idx].unique()
    val_stations = groups.iloc[val_idx].unique()

    print(
        f"Fold {fold}: "
        f"Train stations = {list(train_stations)} | "
        f"Validation station = {list(val_stations)}"
    )

Number of folds: 5
Fold 1: Train stations = ['JALNA_2', 'PAUD_1', 'SONGE_BANGE', 'SUKSALE', 'YELDARI_DAM_1'] | Validation station = ['BHATSANAGAR_1']
Fold 2: Train stations = ['BHATSANAGAR_1', 'JALNA_2', 'PAUD_1', 'SONGE_BANGE', 'YELDARI_DAM_1'] | Validation station = ['SUKSALE']
Fold 3: Train stations = ['BHATSANAGAR_1', 'JALNA_2', 'PAUD_1', 'SONGE_BANGE', 'SUKSALE'] | Validation station = ['YELDARI_DAM_1']
Fold 4: Train stations = ['BHATSANAGAR_1', 'PAUD_1', 'SONGE_BANGE', 'SUKSALE', 'YELDARI_DAM_1'] | Validation station = ['JALNA_2']
Fold 5: Train stations = ['BHATSANAGAR_1', 'JALNA_2', 'SUKSALE', 'YELDARI_DAM_1'] | Validation station = ['PAUD_1', 'SONGE_BANGE']


In [5]:
target = "temperature"

y = df[target_cols[target]]

model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="neg_mean_absolute_error",
    cv=group_kfold,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X, y, groups=groups)

print("\nBest parameters:")
print(search.best_params_)

print("\nBest CV MAE:")
print(-search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best parameters:
{'subsample': 1.0, 'reg_lambda': 10, 'reg_alpha': 1, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 8, 'learning_rate': 0.01, 'colsample_bytree': 0.7}

Best CV MAE:
2.503300957177721


In [6]:
# Evaluate tuned temperature model using LOSO

temperature_results = []

for test_station in groups.unique():

    train_mask = df["station_id"] != test_station
    test_mask = df["station_id"] == test_station

    X_train = df.loc[train_mask, feature_cols]
    X_test = df.loc[test_mask, feature_cols]

    y_train = df.loc[train_mask, target_cols["temperature"]]

    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        **search.best_params_,
    )

    model.fit(X_train, y_train)

    predicted_residual = model.predict(X_test)

    # Reconstruct final temperature prediction
    physics_prediction = df.loc[
        test_mask, "physics_temperature"
    ].values

    actual_observation = df.loc[
        test_mask, "temperature_obs"
    ].values

    final_prediction = physics_prediction + predicted_residual

    physics_mae = mean_absolute_error(
        actual_observation,
        physics_prediction
    )

    ml_mae = mean_absolute_error(
        actual_observation,
        final_prediction
    )

    physics_rmse = np.sqrt(
        mean_squared_error(
            actual_observation,
            physics_prediction
        )
    )

    ml_rmse = np.sqrt(
        mean_squared_error(
            actual_observation,
            final_prediction
        )
    )

    improvement = (
        (physics_mae - ml_mae)
        / physics_mae
        * 100
    )

    temperature_results.append({
        "station": test_station,
        "physics_mae": physics_mae,
        "ml_mae": ml_mae,
        "physics_rmse": physics_rmse,
        "ml_rmse": ml_rmse,
        "improvement_percent": improvement,
    })

temperature_results_df = pd.DataFrame(temperature_results)

print(temperature_results_df.to_string(index=False))

print("\nAverage:")
print(temperature_results_df.mean(numeric_only=True))

      station  physics_mae   ml_mae  physics_rmse  ml_rmse  improvement_percent
BHATSANAGAR_1     4.464921 2.591220      5.747046 3.176945            41.964930
      JALNA_2     4.631700 2.229876      5.824797 3.062185            51.856202
       PAUD_1     4.116500 1.639911      6.003974 2.335587            60.162483
  SONGE_BANGE     4.609860 1.800886      6.273466 2.594289            60.934042
      SUKSALE     4.326208 2.640290      6.021810 3.447384            38.969867
YELDARI_DAM_1     4.210536 2.519716      5.406307 3.491217            40.156878

Average:
physics_mae             4.393287
ml_mae                  2.236983
physics_rmse            5.879567
ml_rmse                 3.017934
improvement_percent    49.007401
dtype: float64


In [7]:
target = "humidity"

y = df[target_cols[target]]

model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

search_humidity = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="neg_mean_absolute_error",
    cv=group_kfold,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search_humidity.fit(X, y, groups=groups)

print("\nBest parameters:")
print(search_humidity.best_params_)

print("\nBest CV MAE:")
print(-search_humidity.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best parameters:
{'subsample': 0.9, 'reg_lambda': 10, 'reg_alpha': 1, 'n_estimators': 1000, 'min_child_weight': 10, 'max_depth': 10, 'learning_rate': 0.01, 'colsample_bytree': 0.9}

Best CV MAE:
7.307823722977953


In [8]:
# Evaluate tuned humidity model using LOSO

humidity_results = []

for test_station in groups.unique():

    train_mask = df["station_id"] != test_station
    test_mask = df["station_id"] == test_station

    X_train = df.loc[train_mask, feature_cols]
    X_test = df.loc[test_mask, feature_cols]

    y_train = df.loc[
        train_mask,
        target_cols["humidity"]
    ]

    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        **search_humidity.best_params_,
    )

    model.fit(X_train, y_train)

    predicted_residual = model.predict(X_test)

    physics_prediction = df.loc[
        test_mask,
        "physics_relative_humidity"
    ].values

    actual_observation = df.loc[
        test_mask,
        "humidity_obs"
    ].values

    final_prediction = physics_prediction + predicted_residual

    physics_mae = mean_absolute_error(
        actual_observation,
        physics_prediction
    )

    ml_mae = mean_absolute_error(
        actual_observation,
        final_prediction
    )

    physics_rmse = np.sqrt(
        mean_squared_error(
            actual_observation,
            physics_prediction
        )
    )

    ml_rmse = np.sqrt(
        mean_squared_error(
            actual_observation,
            final_prediction
        )
    )

    improvement = (
        (physics_mae - ml_mae)
        / physics_mae
        * 100
    )

    humidity_results.append({
        "station": test_station,
        "physics_mae": physics_mae,
        "ml_mae": ml_mae,
        "physics_rmse": physics_rmse,
        "ml_rmse": ml_rmse,
        "improvement_percent": improvement,
    })

humidity_results_df = pd.DataFrame(humidity_results)

print(humidity_results_df.to_string(index=False))

print("\nAverage:")
print(humidity_results_df.mean(numeric_only=True))

      station  physics_mae    ml_mae  physics_rmse   ml_rmse  improvement_percent
BHATSANAGAR_1    15.750792  5.838889     20.941105  8.102314            62.929554
      JALNA_2    20.580884  5.454840     25.648457  7.935475            73.495597
       PAUD_1    15.107084  6.803156     20.487452  8.989218            54.967111
  SONGE_BANGE    22.498168 17.651844     29.777724 21.343414            21.540970
      SUKSALE    18.524217  8.380870     24.377511 10.403885            54.757221
YELDARI_DAM_1    19.691163  6.090383     24.380555  8.626365            69.070477

Average:
physics_mae            18.692051
ml_mae                  8.369997
physics_rmse           24.268801
ml_rmse                10.900112
improvement_percent    56.126822
dtype: float64


In [9]:
target = "wind"

y = df[target_cols[target]]

model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

search_wind = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="neg_mean_absolute_error",
    cv=group_kfold,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search_wind.fit(X, y, groups=groups)

print("\nBest parameters:")
print(search_wind.best_params_)

print("\nBest CV MAE:")
print(-search_wind.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best parameters:
{'subsample': 1.0, 'reg_lambda': 10, 'reg_alpha': 1, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 8, 'learning_rate': 0.01, 'colsample_bytree': 0.7}

Best CV MAE:
0.6839519952608336


In [10]:
# Evaluate tuned wind model using LOSO

wind_results = []

for test_station in groups.unique():

    train_mask = df["station_id"] != test_station
    test_mask = df["station_id"] == test_station

    X_train = df.loc[train_mask, feature_cols]
    X_test = df.loc[test_mask, feature_cols]

    y_train = df.loc[
        train_mask,
        target_cols["wind"]
    ]

    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        **search_wind.best_params_,
    )

    model.fit(X_train, y_train)

    predicted_residual = model.predict(X_test)

    physics_prediction = df.loc[
        test_mask,
        "physics_wind_speed"
    ].values

    actual_observation = df.loc[
        test_mask,
        "wind_speed_obs"
    ].values

    final_prediction = physics_prediction + predicted_residual

    physics_mae = mean_absolute_error(
        actual_observation,
        physics_prediction
    )

    ml_mae = mean_absolute_error(
        actual_observation,
        final_prediction
    )

    physics_rmse = np.sqrt(
        mean_squared_error(
            actual_observation,
            physics_prediction
        )
    )

    ml_rmse = np.sqrt(
        mean_squared_error(
            actual_observation,
            final_prediction
        )
    )

    improvement = (
        (physics_mae - ml_mae)
        / physics_mae
        * 100
    )

    wind_results.append({
        "station": test_station,
        "physics_mae": physics_mae,
        "ml_mae": ml_mae,
        "physics_rmse": physics_rmse,
        "ml_rmse": ml_rmse,
        "improvement_percent": improvement,
    })

wind_results_df = pd.DataFrame(wind_results)

print(wind_results_df.to_string(index=False))

print("\nAverage:")
print(wind_results_df.mean(numeric_only=True))

      station  physics_mae   ml_mae  physics_rmse  ml_rmse  improvement_percent
BHATSANAGAR_1     1.428183 0.694971      1.809132 0.924768            51.338801
      JALNA_2     1.944524 0.468613      2.299961 0.669239            75.900883
       PAUD_1     1.329918 0.963859      1.677398 1.382912            27.524928
  SONGE_BANGE     1.803674 0.903604      2.275879 1.173716            49.902019
      SUKSALE     1.781924 0.614266      2.128736 0.866798            65.527962
YELDARI_DAM_1     1.930305 0.707317      2.264735 0.912189            63.357230

Average:
physics_mae             1.703088
ml_mae                  0.725438
physics_rmse            2.075974
ml_rmse                 0.988270
improvement_percent    55.591970
dtype: float64


In [11]:
# ============================================================
# Feature groups for ablation study
# ============================================================

era5_features = [
    "latitude",
    "longitude",
    "era5_temperature",
    "era5_dewpoint_temperature",
    "era5_u10",
    "era5_v10",
    "era5_relative_humidity",
    "era5_wind_speed",
    "era5_wind_direction",
    "hour",
    "month",
    "day_of_year",
]

physics_features = [
    "station_elevation_m",
    "era5_elevation_m",
    "elevation_difference_m",
    "physics_temperature",
    "physics_relative_humidity",
    "physics_wind_speed",
    "humidity_clamp_flag",
]

full_features = era5_features + physics_features

print("ERA5 features:", len(era5_features))
print("Physics features:", len(physics_features))
print("Full features:", len(full_features))

ERA5 features: 12
Physics features: 7
Full features: 19


In [12]:
# ============================================================
# ERA5-only feature ablation
# ============================================================

era5_results = []

for test_station in groups.unique():

    train_mask = df["station_id"] != test_station
    test_mask = df["station_id"] == test_station

    X_train = df.loc[train_mask, era5_features]
    X_test = df.loc[test_mask, era5_features]

    for variable, target_col in target_cols.items():

        model = XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            objective="reg:squarederror",
            n_jobs=-1
        )

        model.fit(
            X_train,
            df.loc[train_mask, target_col]
        )

        predicted_residual = model.predict(X_test)

        if variable == "temperature":
            physics_col = "physics_temperature"
            observation_col = "temperature_obs"

        elif variable == "humidity":
            physics_col = "physics_relative_humidity"
            observation_col = "humidity_obs"

        else:
            physics_col = "physics_wind_speed"
            observation_col = "wind_speed_obs"

        physics_prediction = df.loc[
            test_mask, physics_col
        ].values

        observation = df.loc[
            test_mask, observation_col
        ].values

        final_prediction = (
            physics_prediction + predicted_residual
        )

        physics_mae = mean_absolute_error(
            observation,
            physics_prediction
        )

        ml_mae = mean_absolute_error(
            observation,
            final_prediction
        )

        improvement = (
            (physics_mae - ml_mae)
            / physics_mae
            * 100
        )

        era5_results.append({
            "station": test_station,
            "variable": variable,
            "physics_mae": physics_mae,
            "era5_only_ml_mae": ml_mae,
            "improvement_percent": improvement
        })


era5_results_df = pd.DataFrame(era5_results)

print(era5_results_df.to_string(index=False))

print("\nAverage:")
print(
    era5_results_df
    .groupby("variable")
    .mean(numeric_only=True)
)

      station    variable  physics_mae  era5_only_ml_mae  improvement_percent
BHATSANAGAR_1 temperature     4.464921          2.898724            35.077830
BHATSANAGAR_1    humidity    15.750792          6.218850            60.517221
BHATSANAGAR_1        wind     1.428183          0.750857            47.425705
      JALNA_2 temperature     4.631700          2.694592            41.822831
      JALNA_2    humidity    20.580884          7.047467            65.757221
      JALNA_2        wind     1.944524          0.555248            71.445543
       PAUD_1 temperature     4.116500          1.406718            65.827319
       PAUD_1    humidity    15.107084          7.160460            52.601972
       PAUD_1        wind     1.329918          0.938903            29.401468
  SONGE_BANGE temperature     4.609860          1.919393            58.363299
  SONGE_BANGE    humidity    22.498168         19.178500            14.755280
  SONGE_BANGE        wind     1.803674          0.994277        

In [ ]:
# Full features WITHOUT latitude and longitude

no_location_features = [
    col for col in full_features
    if col not in ["latitude", "longitude"]
]

print("Features:", len(no_location_features))
print(no_location_features)

Features: 17
['era5_temperature', 'era5_dewpoint_temperature', 'era5_u10', 'era5_v10', 'era5_relative_humidity', 'era5_wind_speed', 'era5_wind_direction', 'hour', 'month', 'day_of_year', 'station_elevation_m', 'era5_elevation_m', 'elevation_difference_m', 'physics_temperature', 'physics_relative_humidity', 'physics_wind_speed', 'humidity_clamp_flag']


In [14]:
# LOSO: Full features WITHOUT latitude/longitude

no_location_results = []

for test_station in groups.unique():

    train_mask = df["station_id"] != test_station
    test_mask = df["station_id"] == test_station

    X_train = df.loc[train_mask, no_location_features]
    X_test = df.loc[test_mask, no_location_features]

    for variable, target_col in target_cols.items():

        model = XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            objective="reg:squarederror",
            n_jobs=-1
        )

        model.fit(
            X_train,
            df.loc[train_mask, target_col]
        )

        predicted_residual = model.predict(X_test)

        if variable == "temperature":
            physics_col = "physics_temperature"
            observation_col = "temperature_obs"

        elif variable == "humidity":
            physics_col = "physics_relative_humidity"
            observation_col = "humidity_obs"

        else:
            physics_col = "physics_wind_speed"
            observation_col = "wind_speed_obs"

        physics_prediction = df.loc[
            test_mask, physics_col
        ].values

        observation = df.loc[
            test_mask, observation_col
        ].values

        final_prediction = (
            physics_prediction + predicted_residual
        )

        physics_mae = mean_absolute_error(
            observation,
            physics_prediction
        )

        ml_mae = mean_absolute_error(
            observation,
            final_prediction
        )

        physics_rmse = np.sqrt(
            mean_squared_error(
                observation,
                physics_prediction
            )
        )

        ml_rmse = np.sqrt(
            mean_squared_error(
                observation,
                final_prediction
            )
        )

        improvement = (
            (physics_mae - ml_mae)
            / physics_mae
            * 100
        )

        no_location_results.append({
            "station": test_station,
            "variable": variable,
            "physics_mae": physics_mae,
            "ml_mae": ml_mae,
            "physics_rmse": physics_rmse,
            "ml_rmse": ml_rmse,
            "improvement_percent": improvement
        })


no_location_results_df = pd.DataFrame(no_location_results)

print(no_location_results_df.to_string(index=False))

print("\nAverage:")
print(
    no_location_results_df
    .groupby("variable")
    .mean(numeric_only=True)
)

      station    variable  physics_mae    ml_mae  physics_rmse   ml_rmse  improvement_percent
BHATSANAGAR_1 temperature     4.464921  2.515666      5.747046  3.128227            43.657105
BHATSANAGAR_1    humidity    15.750792  6.455822     20.941105  8.959573            59.012713
BHATSANAGAR_1        wind     1.428183  0.655606      1.809132  0.900747            54.095087
      JALNA_2 temperature     4.631700  1.954342      5.824797  2.802614            57.805073
      JALNA_2    humidity    20.580884  5.673968     25.648457  8.005633            72.430884
      JALNA_2        wind     1.944524  0.305789      2.299961  0.526348            84.274372
       PAUD_1 temperature     4.116500  1.845738      6.003974  2.526935            55.162442
       PAUD_1    humidity    15.107084 16.164755     20.487452 19.334993            -7.001156
       PAUD_1        wind     1.329918  1.320305      1.677398  1.800864             0.722856
  SONGE_BANGE temperature     4.609860  1.613936      6.2734

In [15]:
# ============================================================
# PHYSICS-ONLY FEATURE ABLATION
# ============================================================

physics_only_features = [
    "station_elevation_m",
    "era5_elevation_m",
    "elevation_difference_m",
    "physics_temperature",
    "physics_relative_humidity",
    "physics_wind_speed",
    "humidity_clamp_flag",
]

print("Number of physics-only features:", len(physics_only_features))
print(physics_only_features)


# ============================================================
# LOSO validation
# ============================================================

physics_only_results = []

for test_station in groups.unique():

    train_mask = df["station_id"] != test_station
    test_mask = df["station_id"] == test_station

    X_train = df.loc[
        train_mask,
        physics_only_features
    ]

    X_test = df.loc[
        test_mask,
        physics_only_features
    ]

    for variable, target_col in target_cols.items():

        print(
            f"Training {variable} | "
            f"Test station: {test_station}"
        )

        model = XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            objective="reg:squarederror",
            n_jobs=-1
        )

        model.fit(
            X_train,
            df.loc[train_mask, target_col]
        )

        predicted_residual = model.predict(X_test)

        # ----------------------------------------------------
        # Select physics prediction and observation
        # ----------------------------------------------------

        if variable == "temperature":
            physics_col = "physics_temperature"
            observation_col = "temperature_obs"

        elif variable == "humidity":
            physics_col = "physics_relative_humidity"
            observation_col = "humidity_obs"

        else:
            physics_col = "physics_wind_speed"
            observation_col = "wind_speed_obs"

        physics_prediction = df.loc[
            test_mask,
            physics_col
        ].values

        observation = df.loc[
            test_mask,
            observation_col
        ].values

        # ----------------------------------------------------
        # Reconstruct ML-corrected prediction
        # ----------------------------------------------------

        final_prediction = (
            physics_prediction + predicted_residual
        )

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        physics_mae = mean_absolute_error(
            observation,
            physics_prediction
        )

        ml_mae = mean_absolute_error(
            observation,
            final_prediction
        )

        physics_rmse = np.sqrt(
            mean_squared_error(
                observation,
                physics_prediction
            )
        )

        ml_rmse = np.sqrt(
            mean_squared_error(
                observation,
                final_prediction
            )
        )

        improvement = (
            (physics_mae - ml_mae)
            / physics_mae
            * 100
        )

        physics_only_results.append({
            "station": test_station,
            "variable": variable,
            "physics_mae": physics_mae,
            "ml_mae": ml_mae,
            "physics_rmse": physics_rmse,
            "ml_rmse": ml_rmse,
            "improvement_percent": improvement
        })


# ============================================================
# Results
# ============================================================

physics_only_results_df = pd.DataFrame(
    physics_only_results
)

print("\n")
print("=" * 80)
print("PHYSICS-ONLY FEATURE RESULTS")
print("=" * 80)

print(
    physics_only_results_df.to_string(index=False)
)


# ============================================================
# Average results
# ============================================================

print("\n")
print("=" * 80)
print("AVERAGE RESULTS")
print("=" * 80)

physics_only_summary = (
    physics_only_results_df
    .groupby("variable")
    .agg({
        "physics_mae": "mean",
        "ml_mae": "mean",
        "physics_rmse": "mean",
        "ml_rmse": "mean",
        "improvement_percent": "mean"
    })
)

print(physics_only_summary)

Number of physics-only features: 7
['station_elevation_m', 'era5_elevation_m', 'elevation_difference_m', 'physics_temperature', 'physics_relative_humidity', 'physics_wind_speed', 'humidity_clamp_flag']
Training temperature | Test station: BHATSANAGAR_1
Training humidity | Test station: BHATSANAGAR_1
Training wind | Test station: BHATSANAGAR_1
Training temperature | Test station: JALNA_2
Training humidity | Test station: JALNA_2
Training wind | Test station: JALNA_2
Training temperature | Test station: PAUD_1
Training humidity | Test station: PAUD_1
Training wind | Test station: PAUD_1
Training temperature | Test station: SONGE_BANGE
Training humidity | Test station: SONGE_BANGE
Training wind | Test station: SONGE_BANGE
Training temperature | Test station: SUKSALE
Training humidity | Test station: SUKSALE
Training wind | Test station: SUKSALE
Training temperature | Test station: YELDARI_DAM_1
Training humidity | Test station: YELDARI_DAM_1
Training wind | Test station: YELDARI_DAM_1


P

In [16]:
from xgboost import XGBRegressor
import pandas as pd
import matplotlib.pyplot as plt

# Original baseline XGBoost parameters
model_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "objective": "reg:squarederror",
    "n_jobs": -1
}

target_cols = {
    "temperature": "temperature_residual",
    "humidity": "humidity_residual",
    "wind": "wind_speed_residual",
}

feature_groups = {
    "full": full_features,
    "era5_only": era5_features,
    "physics_only": physics_features,
}

# Train on all available stations for feature-importance analysis
X = df[full_features]

feature_importance = {}

for variable, target in target_cols.items():
    print(f"Training {variable} model...")

    model = XGBRegressor(**model_params)
    model.fit(X, df[target])

    importance = pd.Series(
        model.feature_importances_,
        index=full_features
    ).sort_values(ascending=False)

    feature_importance[variable] = importance

    print(f"\n{variable.upper()} feature importance:")
    print(importance.to_string())
    print("\n" + "=" * 60)

Training temperature model...

TEMPERATURE feature importance:
hour                         0.329857
physics_relative_humidity    0.241103
elevation_difference_m       0.093452
physics_temperature          0.040223
longitude                    0.038792
era5_relative_humidity       0.035781
station_elevation_m          0.032664
day_of_year                  0.031279
era5_wind_speed              0.022233
era5_wind_direction          0.021279
era5_temperature             0.018419
latitude                     0.017218
era5_v10                     0.015961
month                        0.014761
era5_u10                     0.013858
physics_wind_speed           0.011059
era5_dewpoint_temperature    0.010239
era5_elevation_m             0.009968
humidity_clamp_flag          0.001854

Training humidity model...

HUMIDITY feature importance:
longitude                    0.224572
hour                         0.220383
physics_relative_humidity    0.207969
station_elevation_m          0.054864
eleva

In [19]:
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
import pandas as pd
import numpy as np

model_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "objective": "reg:squarederror",
    "n_jobs": -1
}

target = "wind_speed_residual"

stations = df["station_id"].unique()

all_importances = []

for test_station in stations:

    train_df = df[df["station_id"] != test_station]
    test_df = df[df["station_id"] == test_station]

    X_train = train_df[full_features]
    y_train = train_df[target]

    X_test = test_df[full_features]
    y_test = test_df[target]

    model = XGBRegressor(**model_params)
    model.fit(X_train, y_train)

    result = permutation_importance(
        model,
        X_test,
        y_test,
        scoring="neg_mean_absolute_error",
        n_repeats=5,
        random_state=42,
        n_jobs=-1
    )

    importance = pd.Series(
        result.importances_mean,
        index=full_features
    )

    importance.name = test_station
    all_importances.append(importance)

importance_df = pd.DataFrame(all_importances)

# Average importance across all held-out stations
mean_importance = importance_df.mean(axis=0).sort_values(ascending=False)

print("LOSO PERMUTATION IMPORTANCE - WIND")
print("=" * 60)
print(mean_importance.to_string())

LOSO PERMUTATION IMPORTANCE - WIND
era5_wind_speed              0.375468
physics_wind_speed           0.159674
hour                         0.120227
physics_temperature          0.028267
era5_temperature             0.026824
era5_relative_humidity       0.020106
era5_u10                     0.019819
day_of_year                  0.015129
physics_relative_humidity    0.011464
era5_v10                     0.009666
era5_dewpoint_temperature    0.009188
month                        0.008322
era5_wind_direction          0.001936
humidity_clamp_flag          0.000089
latitude                     0.000000
longitude                    0.000000
elevation_difference_m       0.000000
station_elevation_m          0.000000
era5_elevation_m             0.000000


TRAINING MODEL ON REDUCED FEATURES

In [20]:
# Reduced feature sets based on LOSO permutation importance

reduced_features = {
    "temperature": [
        "era5_temperature",
        "era5_dewpoint_temperature",
        "era5_relative_humidity",
        "physics_temperature",
        "physics_relative_humidity",
        "hour",
        "day_of_year",
        "month",
    ],

    "humidity": [
        "era5_relative_humidity",
        "era5_dewpoint_temperature",
        "era5_temperature",
        "physics_relative_humidity",
        "physics_temperature",
        "hour",
        "day_of_year",
        "month",
    ],

    "wind": [
        "era5_wind_speed",
        "era5_u10",
        "era5_v10",
        "era5_wind_direction",
        "physics_wind_speed",
        "hour",
        "day_of_year",
        "month",
    ]
}

for variable, features in reduced_features.items():
    print(f"{variable}: {len(features)} features")
    print(features)
    print()

temperature: 8 features
['era5_temperature', 'era5_dewpoint_temperature', 'era5_relative_humidity', 'physics_temperature', 'physics_relative_humidity', 'hour', 'day_of_year', 'month']

humidity: 8 features
['era5_relative_humidity', 'era5_dewpoint_temperature', 'era5_temperature', 'physics_relative_humidity', 'physics_temperature', 'hour', 'day_of_year', 'month']

wind: 8 features
['era5_wind_speed', 'era5_u10', 'era5_v10', 'era5_wind_direction', 'physics_wind_speed', 'hour', 'day_of_year', 'month']



In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import pandas as pd
import numpy as np

model_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "objective": "reg:squarederror",
    "n_jobs": -1
}

target_cols = {
    "temperature": "temperature_residual",
    "humidity": "humidity_residual",
    "wind": "wind_speed_residual",
}

results = []

stations = df["station_id"].unique()

for variable, target in target_cols.items():

    features = reduced_features[variable]

    print(f"\n{'=' * 70}")
    print(f"VARIABLE: {variable.upper()}")
    print(f"FEATURES: {len(features)}")
    print(f"{'=' * 70}")

    for test_station in stations:

        train_df = df[df["station_id"] != test_station]
        test_df = df[df["station_id"] == test_station]

        X_train = train_df[features]
        y_train = train_df[target]

        X_test = test_df[features]
        y_test = test_df[target]

        model = XGBRegressor(**model_params)
        model.fit(X_train, y_train)

        predicted_residual = model.predict(X_test)

        # Final prediction = physics baseline + predicted residual
        if variable == "temperature":
            baseline = test_df["physics_temperature"]
        elif variable == "humidity":
            baseline = test_df["physics_relative_humidity"]
        else:
            baseline = test_df["physics_wind_speed"]

        y_pred = baseline + predicted_residual

        # Observed value
        if variable == "temperature":
            y_true = test_df["temperature_obs"]
        elif variable == "humidity":
            y_true = test_df["humidity_obs"]
        else:
            y_true = test_df["wind_speed_obs"]

        # Physics-only baseline
        physics_mae = mean_absolute_error(y_true, baseline)
        physics_rmse = np.sqrt(mean_squared_error(y_true, baseline))

        # ML-corrected prediction
        ml_mae = mean_absolute_error(y_true, y_pred)
        ml_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        improvement = (
            (physics_mae - ml_mae) / physics_mae
        ) * 100

        results.append({
            "test_station": test_station,
            "variable": variable,
            "physics_mae": physics_mae,
            "ml_mae": ml_mae,
            "physics_rmse": physics_rmse,
            "ml_rmse": ml_rmse,
            "improvement_percent": improvement
        })

        print(
            f"{test_station:20s} "
            f"MAE: {ml_mae:.4f} | "
            f"RMSE: {ml_rmse:.4f} | "
            f"Improvement: {improvement:.2f}%"
        )


results_df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("REDUCED FEATURE LOSO RESULTS")
print("=" * 80)

summary = results_df.groupby("variable").agg({
    "physics_mae": "mean",
    "ml_mae": "mean",
    "physics_rmse": "mean",
    "ml_rmse": "mean",
    "improvement_percent": "mean"
})

print(summary)


VARIABLE: TEMPERATURE
FEATURES: 8
BHATSANAGAR_1        MAE: 2.5977 | RMSE: 3.2053 | Improvement: 41.82%
JALNA_2              MAE: 2.0221 | RMSE: 3.1374 | Improvement: 56.34%
PAUD_1               MAE: 1.8654 | RMSE: 2.3381 | Improvement: 54.69%
SONGE_BANGE          MAE: 2.0102 | RMSE: 2.8193 | Improvement: 56.39%
SUKSALE              MAE: 1.9918 | RMSE: 2.4962 | Improvement: 53.96%
YELDARI_DAM_1        MAE: 2.5555 | RMSE: 3.5744 | Improvement: 39.31%

VARIABLE: HUMIDITY
FEATURES: 8
BHATSANAGAR_1        MAE: 10.2758 | RMSE: 14.0273 | Improvement: 34.76%
JALNA_2              MAE: 11.5427 | RMSE: 14.2898 | Improvement: 43.92%
PAUD_1               MAE: 7.6539 | RMSE: 10.0744 | Improvement: 49.34%
SONGE_BANGE          MAE: 10.9505 | RMSE: 15.6644 | Improvement: 51.33%
SUKSALE              MAE: 9.6362 | RMSE: 13.2879 | Improvement: 47.98%
YELDARI_DAM_1        MAE: 8.9805 | RMSE: 11.9641 | Improvement: 54.39%

VARIABLE: WIND
FEATURES: 8
BHATSANAGAR_1        MAE: 0.6622 | RMSE: 0.8994 | Improv

In [22]:
# Compare FULL vs TARGETED feature sets using LOSO

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import pandas as pd
import numpy as np

# Same baseline XGBoost parameters
model_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "objective": "reg:squarederror",
    "n_jobs": -1
}

target_cols = {
    "temperature": "temperature_residual",
    "humidity": "humidity_residual",
    "wind": "wind_speed_residual",
}

# Targeted feature sets
targeted_features = {
    "temperature": [
        "era5_temperature",
        "era5_dewpoint_temperature",
        "era5_relative_humidity",
        "physics_temperature",
        "physics_relative_humidity",
        "hour",
        "day_of_year",
    ],

    "humidity": [
        "era5_relative_humidity",
        "era5_dewpoint_temperature",
        "era5_temperature",
        "physics_relative_humidity",
        "physics_temperature",
        "hour",
        "day_of_year",
        "month",
        "latitude",
        "longitude",
        "station_elevation_m",
        "elevation_difference_m",
    ],

    "wind": [
        "era5_wind_speed",
        "era5_u10",
        "era5_v10",
        "era5_wind_direction",
        "physics_wind_speed",
        "hour",
        "day_of_year",
        "month",
    ]
}

# Full feature set
full_feature_sets = {
    "temperature": full_features,
    "humidity": full_features,
    "wind": full_features,
}

results = []

stations = df["station_id"].unique()

for variable, target in target_cols.items():

    print(f"\n{'=' * 70}")
    print(f"VARIABLE: {variable.upper()}")
    print(f"{'=' * 70}")

    for feature_set_name, feature_set in [
        ("full", full_feature_sets[variable]),
        ("targeted", targeted_features[variable])
    ]:

        print(f"\nTesting {feature_set_name} ({len(feature_set)} features)...")

        for test_station in stations:

            train_df = df[df["station_id"] != test_station]
            test_df = df[df["station_id"] == test_station]

            X_train = train_df[feature_set]
            y_train = train_df[target]

            X_test = test_df[feature_set]
            y_test = test_df[target]

            model = XGBRegressor(**model_params)
            model.fit(X_train, y_train)

            predicted_residual = model.predict(X_test)

            # Physics baseline
            if variable == "temperature":
                baseline = test_df["physics_temperature"]
                y_true = test_df["temperature_obs"]

            elif variable == "humidity":
                baseline = test_df["physics_relative_humidity"]
                y_true = test_df["humidity_obs"]

            else:
                baseline = test_df["physics_wind_speed"]
                y_true = test_df["wind_speed_obs"]

            # Final ML prediction
            y_pred = baseline + predicted_residual

            ml_mae = mean_absolute_error(y_true, y_pred)
            ml_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

            physics_mae = mean_absolute_error(y_true, baseline)
            physics_rmse = np.sqrt(mean_squared_error(y_true, baseline))

            improvement = (
                (physics_mae - ml_mae) / physics_mae
            ) * 100

            results.append({
                "variable": variable,
                "feature_set": feature_set_name,
                "test_station": test_station,
                "n_features": len(feature_set),
                "physics_mae": physics_mae,
                "ml_mae": ml_mae,
                "physics_rmse": physics_rmse,
                "ml_rmse": ml_rmse,
                "improvement_percent": improvement
            })


results_df = pd.DataFrame(results)

# Average across the six held-out stations
comparison = (
    results_df
    .groupby(["variable", "feature_set"])
    .agg({
        "n_features": "first",
        "physics_mae": "mean",
        "ml_mae": "mean",
        "physics_rmse": "mean",
        "ml_rmse": "mean",
        "improvement_percent": "mean"
    })
    .reset_index()
)

print("\n" + "=" * 90)
print("FULL vs TARGETED FEATURE COMPARISON")
print("=" * 90)

print(comparison.to_string(index=False))


VARIABLE: TEMPERATURE

Testing full (19 features)...

Testing targeted (7 features)...

VARIABLE: HUMIDITY

Testing full (19 features)...

Testing targeted (12 features)...

VARIABLE: WIND

Testing full (19 features)...

Testing targeted (8 features)...

FULL vs TARGETED FEATURE COMPARISON
   variable feature_set  n_features  physics_mae   ml_mae  physics_rmse   ml_rmse  improvement_percent
   humidity        full          19    18.692051 8.460627     24.268801 10.977760            55.736498
   humidity    targeted          12    18.692051 8.663878     24.268801 11.177623            54.553054
temperature        full          19     4.393287 2.241970      5.879567  3.032162            48.877375
temperature    targeted           7     4.393287 2.160895      5.879567  2.909020            50.705065
       wind        full          19     1.703088 0.747197      2.075974  1.012930            54.323967
       wind    targeted           8     1.703088 0.747172      2.075974  1.035339         